In [1]:
import pdfplumber, re, json, os

In [2]:
pdf = pdfplumber.open("Road-User-Handbook-English.pdf")
print("pages:", len(pdf.pages))

pages: 212


In [3]:
def clean_text(text):
    """Strip page furniture and normalise whitespace."""
    if not text:
        return ""
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        if re.fullmatch(r"\d{1,3}", s):                  # bare page numbers
            continue
        if re.search(r"road user handbook", s, re.I):    # repeated footer
            continue
        lines.append(s)
    return " ".join(lines)

print(clean_text(pdf.pages[20].extract_text())[:400])

Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions are in place to help keep you safe as you develop your driving skills. They also apply when you drive in another state or territory. If you do not follow these restrictions, you’ll get a fine. You can a


In [4]:
pages = []
for i, page in enumerate(pdf.pages, start=1):
    cleaned = clean_text(page.extract_text())
    if len(cleaned.split()) < 30:      # drop image-only pages
        continue
    pages.append({"page": i, "text": cleaned})

print(f"kept {len(pages)} of {len(pdf.pages)} pages")

kept 208 of 212 pages


In [5]:
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/handbook_pages.jsonl", "w") as f:
    for p in pages:
        f.write(json.dumps(p) + "\n")
print("written")

written


In [6]:
def chunk_pages(pages, size_words=150, overlap_words=30):
    chunks, cid = [], 0
    for p in pages:
        words = p["text"].split()
        start = 0
        while start < len(words):
            window = words[start:start + size_words]
            chunks.append({
                "chunk_id": f"c{cid:04d}",
                "page": p["page"],
                "text": " ".join(window),
            })
            cid += 1
            if start + size_words >= len(words):
                break
            start += (size_words - overlap_words)
    return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks")

348 chunks


In [7]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/chroma")
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

try:
    client.delete_collection("handbook")
except Exception:
    pass

col = client.create_collection(name="handbook", embedding_function=ef)
col.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)
print("indexed:", col.count())

c:\Users\chris\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2817.34it/s]


indexed: 348


In [8]:
res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=5)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"[p.{meta['page']}] {doc[:200]}\n")

[p.31] Licences Alcohol and drugs When supervising a learning driver, a supervisor must: • have a blood alcohol concentration (BAC) under 0.05 • not have illegal drugs present in their system or be under the

[p.41] Safe driving behaviour Drive to road conditions Even if you’re driving at or below the speed limit, you may be driving too fast for road conditions such as curves, rain, heavy traffic or night-time. A

[p.199] driving over the alcohol limit (under 0.08, first offence) • not paying a fine • being medically unfit to drive. For some offences, police can suspend and take away your licence immediately. These inc

[p.201] Penalties Drink-driving offences that can result in an interlock include: • driving with a blood alcohol concentration of 0.08 or over • any repeat drink-driving offence • driving under the influence 

[p.21] Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisiona

In [9]:
# Failure case F1: the learner BAC chunk exists but ranks poorly
hits = [c for c in chunks if "must be zero" in c["text"]]
print(f"chunks containing the rule: {len(hits)}")
for h in hits:
    print(f"[{h['chunk_id']}] p.{h['page']}: {h['text'][:200]}\n")

res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=10)
target = hits[0]["chunk_id"] if hits else None
for rank, (cid, meta) in enumerate(zip(res["ids"][0], res["metadatas"][0]), start=1):
    mark = "  <-- TARGET" if cid == target else ""
    print(f"{rank}. {cid} p.{meta['page']}{mark}")

chunks containing the rule: 1
[c0031] p.21: Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions ar

1. c0048 p.31
2. c0065 p.41
3. c0324 p.199
4. c0327 p.201
5. c0031 p.21  <-- TARGET
6. c0067 p.42
7. c0069 p.43
8. c0326 p.200
9. c0047 p.30
10. c0051 p.33
